# Data Cleaning

## Objective

The objective of this phase is to prepare the datasets for analysis by validating data types, handling missing values, investigating duplicate records, and improving overall data quality.

## Data Cleaning Tasks

- Load all datasets
- Validate data types
- Convert date columns to datetime
- Handle missing values
- Investigate duplicate records
- Validate cleaned datasets

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Step 1: Validate Data Types

In this step, the data types of each dataset are validated to ensure that every column has the appropriate type before applying any cleaning operations.

In [2]:
import pandas as pd

from retailpulse.utils import (check_data_types , load_datasets )

In [3]:
(
    orders,
    customers,
    geolocation,
    items,
    payments,
    reviews,
    products,
    sellers,
    categories,
) = load_datasets()

In [4]:
check_data_types(orders, "Orders")
check_data_types(products, "Products")
check_data_types(customers, "Customers")
check_data_types(items, "Order Items")
check_data_types(payments, "Order Payments")
check_data_types(reviews, "Order Reviews")
check_data_types(sellers, "Sellers")
check_data_types(categories, "Product Category Name Translation")
check_data_types(geolocation, "Geolocation")

Dataset: Orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
Dataset: Products
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
Dataset: Customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
Dataset: Order Items
order_id                   str
order_item_id            int64
product_id    

### Observations

- Most categorical columns are correctly stored as string (`str`).
- Numerical columns are stored using appropriate numeric data types (`int64` and `float64`).
- All date-related columns are still stored as strings and need to be converted to the `datetime` data type.
- No incorrect data types were identified in the remaining columns.

## Step 2: Convert Date Columns

Date and time columns are currently stored as strings (`str`). In this step, these columns will be converted to the `datetime` data type to enable time-based analysis such as extracting years, months, weekdays, and calculating delivery durations.

In [5]:
from retailpulse.utils import convert_to_datetime

In [6]:
orders_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = convert_to_datetime(orders, orders_date_columns)

In [7]:
items_date_columns = [
    "shipping_limit_date",
]

items = convert_to_datetime(items, items_date_columns)

In [8]:
reviews_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

reviews = convert_to_datetime(reviews, reviews_date_columns)

In [9]:
check_data_types(orders, "Orders")
check_data_types(items, "Order Items")
check_data_types(reviews, "Order Reviews")

Dataset: Orders
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
Dataset: Order Items
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object
Dataset: Order Reviews
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_ans

## Step 3: Handle Missing Values

In this step, missing values are investigated to understand their causes before deciding whether they should be removed, replaced, or retained.

In [11]:
from retailpulse.utils import analyze_missing_values

In [13]:
analyze_missing_values(orders, "Orders")
analyze_missing_values(products, "Products")
analyze_missing_values(reviews, "Order Reviews")
analyze_missing_values(items, "Order Items")
analyze_missing_values(payments, "Order Payments")
analyze_missing_values(sellers, "Sellers")
analyze_missing_values(categories, "Product Category Name Translation")
analyze_missing_values(geolocation, "Geolocation")
analyze_missing_values(customers, "Customers")

Dataset: Orders
                               Missing Values  Percentage (%)
order_delivered_customer_date            2965            2.98
order_delivered_carrier_date             1783            1.79
order_approved_at                         160            0.16
Dataset: Products
                            Missing Values  Percentage (%)
product_category_name                  610            1.85
product_name_lenght                    610            1.85
product_description_lenght             610            1.85
product_photos_qty                     610            1.85
product_weight_g                         2            0.01
product_length_cm                        2            0.01
product_height_cm                        2            0.01
product_width_cm                         2            0.01
Dataset: Order Reviews
                        Missing Values  Percentage (%)
review_comment_title             87656           88.34
review_comment_message           58247           58.70

### Observations

- Most datasets contain no missing values and are ready for analysis.
- Missing values in the Orders dataset are likely related to the order lifecycle (e.g., canceled or undelivered orders) and require further investigation before any cleaning action.
- The Products dataset contains missing values mainly in product-related attributes, suggesting that a small number of products have incomplete information.
- The Reviews dataset contains a high percentage of missing values in the review title and review message columns. This is expected because customers are not required to leave textual feedback when submitting a rating.
- No cleaning actions were applied at this stage because the causes of missing values must be understood before deciding how to handle them.